# 04 — XGBoost: 3-Fold Sliding Window Validation

**Scopo:** confrontare la strategia *sliding window* con l'Expanding Window del notebook 03, mantenendo **le stesse 27 feature e gli stessi hyperparameter** della baseline. In questo esperimento ogni modello usa soltanto i **3 anni precedenti** all'anno di validation, invece di tutto lo storico.

- **Fold 1:** training 2016–2018, validation 2019.
- **Fold 2:** training 2017–2019, validation 2020.
- **Fold 3:** training 2018–2020, validation 2021.
- **Holdout 2022:** dopo i tre fold, un **quarto modello** viene allenato sui casi del 2019–2021 e valutato sul 2022. Non usiamo il 2022 per scegliere la configurazione.

**Gap anti-overlap:** ogni caso rappresenta una finestra di 168 ore a partire da `starttime`. Escludiamo dal training i casi che si sovrappongono alla validation, e dalle validation dei tre fold i casi la cui finestra oltrepassa la fine dell'anno. Lo split avviene per intero caso (`run_no`), mai per singola riga long.

**Nota:** con la sliding window i dati più vecchi **vengono esclusi**, non ricevono un peso inferiore. I casi conservati mantengono lo stesso peso di training, come nella baseline. Per sperimentare finestre di 2 o 4 anni si può modificare `TRAIN_WINDOW_YEARS`.

Il notebook non modifica né sovrascrive i notebook 02 e 03 o i loro risultati.

## 1 — Importazioni e percorso del Parquet

Il notebook cerca la cartella della repository partendo dalla directory di lavoro attuale. Funziona se Jupyter parte da `ml-project-ntnu` o da `Ghigu`.

In [1]:
from pathlib import Path
import json
import time
import time
from tqdm.auto import tqdm
from xgboost.callback import TrainingCallback
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

#questa serve solo per un output figo 
class ProgressCallback(TrainingCallback):
    def __init__(self, total):
        self.pbar = tqdm(total=total, desc="Training XGBoost", unit="albero")

    def after_iteration(self, model, epoch, evals_log):
        self.pbar.update(1)
        return False

    def after_training(self, model):
        self.pbar.close()
        return model



REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "Ghigu" / "data" / "processed").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Cartella Ghigu/data/processed non trovata. "
        "Avvia Jupyter dalla repository ml-project-ntnu o dalla cartella Ghigu."
    )

PROCESSED_DIR = REPO_ROOT / "Ghigu" / "data" / "processed"
DATASET_PATH = PROCESSED_DIR / "feature_table_long.parquet"
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"Dataset non trovato: {DATASET_PATH}. Esegui prima il notebook 01.")

print("Dataset:", DATASET_PATH)

Dataset: /home/ghigu/Scrivania/ml-project-ntnu/Ghigu/data/processed/feature_table_long.parquet


## 2 — Selezione delle feature e caricamento dati

**Le feature sono gli input `X`; `target` è l'etichetta `y`.**

Non usiamo `run_no` (identificativo del caso), `starttime` o `timestamp` come feature: i primi due servono a dividere/ricostruire i dati; il timestamp assoluto per ora non serve. **Conserviamo `hour`**, perché l'ora all'interno della finestra di 168 ore potrebbe essere informativa.

Usiamo `generator` e `plant` come variabili categoriche; XGBoost le gestisce nativamente con `enable_categorical=True`. Sono incluse le **sette feature topologiche** del notebook 01, più le altre caratteristiche numeriche e i vincoli idrici.

In [2]:
BASE_FEATURE_COLUMNS = [
    # Posizione all'interno della settimana e identificazione del generatore
    "hour", "generator", "plant",

    # Prezzo dell'energia
    "price_current",

    # Caratteristiche tecniche del generatore
    "p_min", "p_max", "p_nom", "start_cost", "stop_cost", "penstock",

    # Vincoli idrici e inflow esterno
    "min_volume_Totak", "min_volume_Byrtevatn", "min_volume_Staavatn",
    "min_flow_b_Kjelavatn", "min_flow_b_Hyljelihyl", "min_flow_b_Vinjevatn",
    "min_flow_b_Byrtevatn", "min_flow_b_Bandak",
    "min_flow_r_Vest_Vassdraget", "inflow_r_Vest_Vassdraget",
]

ID_COLUMNS = ["run_no", "starttime", "generator", "hour"]


# ============================================================
# FEATURE ENGINEERING - TOPOLOGY
# ============================================================

TOPOLOGY_FEATURE_COLUMNS = [
    "direct_total_volume",
    "direct_mean_fill_ratio",
    "direct_total_inflow",
    "direct_mean_water_value",
    
    "upstream_total_volume",
    "upstream_total_inflow",
    "upstream_mean_water_value",
]


# ============================================================
# FEATURE ENGINEERING - ELECTRICITY PRICE
# ============================================================

PRICE_FEATURE_COLUMNS = [
    # "price_mean_week", peggiora 
    # "price_std_week",  peggiora 
    # "price_min_week",  peggiora 
    # "price_max_week",  peggiora
    
     "price_diff_week_mean", #migliora
     "price_rank_week",      #migliora 
    
     #"price_delta_prev_hour", peggiora 
     #"price_mean_next_3h",    peggiora 
     "price_mean_next_12h",  #migliora ma è sus 
]

# ============================================================
# FEATURE ENGINEERING - WATER - ELETTRICITY
# ============================================================

WATER_PRICE_FEATURE_COLUMNS = [
    "price_minus_water_value",
    "price_mean_week_minus_water_value",
    #"price_max_week_minus_water_value",
]


# ============================================================
# FEATURE ENGINEERING - INFLOW
# ============================================================

#INFLOW_FEATURE_COLUMNS = [               peggiora, tutte futili 
#    "direct_cumulative_inflow_week",
#     "direct_inflow_std_week",
#     "direct_inflow_max_week",
#]

# ============================================================
# FEATURE ENGINEERING - HISTORICAL INFLOW
# ============================================================

#INFLOW_FEATURE_COLUMNS = [
#      "direct_historical_mean_inflow_for_month",
#     "direct_inflow_anomaly_week",
#     "direct_inflow_ratio_week",
#]



# ============================================================
# FEATURE SELECTION
# ============================================================

FEATURE_COLUMNS = BASE_FEATURE_COLUMNS.copy()

# Feature engineering groups
FEATURE_COLUMNS += TOPOLOGY_FEATURE_COLUMNS
FEATURE_COLUMNS += PRICE_FEATURE_COLUMNS
FEATURE_COLUMNS += WATER_PRICE_FEATURE_COLUMNS

# ============================================================
# COLUMNS TO READ FROM PARQUET
# ============================================================

READ_COLUMNS = list(dict.fromkeys([*ID_COLUMNS, *FEATURE_COLUMNS, "target"]))


# Carichiamo soltanto le colonne che useremo, evitando timestamp.
# praticamente leggiamo dal file indicato da dataset:path, le colonne elencate in Read_Columns
df = pd.read_parquet(DATASET_PATH, columns=READ_COLUMNS)

#df è il datafram con cui lavora pandas, qua diciamo a pandas di trattare starttime come data e ora usando fuso orario UTC
# lo facciamo così possiamo dividere in dati in 80% e 20% in futuro 
df["starttime"] = pd.to_datetime(df["starttime"], utc=True)

# due colonne contengono stringhe e non numeri ( generator e plant ), convertiamo la colonna nel tipo 
# category di Pandas
for column in ["generator", "plant"]:
    df[column] = df[column].astype("category")

# queste sono solo righe di controllo 
assert not df.duplicated(["run_no", "generator", "hour"]).any(), "Chiavi long duplicate"
assert df["target"].notna().all() and df["target"].isin([0, 1]).all(), "Target non binario"


# queste sono solo stampe 
print(f"Dimensioni: {len(df):,} righe × {df.shape[1]} colonne lette")
print(f"Feature utilizzate: {len(READ_COLUMNS)}")
print(f"Casi: {df['run_no'].nunique():,} | Generatori: {df['generator'].nunique()}")
print(f"Target ON: {df['target'].mean():.2%}")
#stampa le prime cinque righe del dataframe per vedere se sono state caricate correttamente
print(df.head(5))

Dimensioni: 6,872,544 righe × 35 colonne lette
Feature utilizzate: 35
Casi: 2,922 | Generatori: 14
Target ON: 51.64%
   run_no                 starttime generator  hour  plant  price_current  \
0       1 2015-01-01 00:00:00+00:00  Hogga_G1     0  Hogga      27.070000   
1       1 2015-01-01 00:00:00+00:00  Hogga_G1     1  Hogga      26.930000   
2       1 2015-01-01 00:00:00+00:00  Hogga_G1     2  Hogga      26.830000   
3       1 2015-01-01 00:00:00+00:00  Hogga_G1     3  Hogga      26.809999   
4       1 2015-01-01 00:00:00+00:00  Hogga_G1     4  Hogga      26.969999   

   p_min  p_max  p_nom  start_cost  ...  direct_mean_water_value  \
0    3.0   17.0   17.0        94.0  ...                     0.01   
1    3.0   17.0   17.0        94.0  ...                     0.01   
2    3.0   17.0   17.0        94.0  ...                     0.01   
3    3.0   17.0   17.0        94.0  ...                     0.01   
4    3.0   17.0   17.0        94.0  ...                     0.01   

   upstream

## 3 — Definizione dei 3 fold temporali e dell'holdout 2022

Costruiamo una tabella con **una riga per caso**, ordinata cronologicamente. Ogni fold usa soltanto i casi compresi nei **3 anni che precedono la validation** (*sliding window*); la finestra avanza di un anno a ogni fold, senza continuare ad accumulare tutto lo storico.

La durata si regola modificando `TRAIN_WINDOW_YEARS = 3`. Per evitare che le finestre di 168 ore condividano le stesse ore tra training e validation, imponiamo `starttime + 168 ore <= primo giorno della validation`. Inoltre, nei tre fold, valutiamo solo i casi la cui finestra termina **entro il primo gennaio dell'anno successivo**. Il test sul 2022 comprende invece anche gli ultimi sei giorni dell'anno.

Manteniamo **gli stessi anni di validation e lo stesso gap** dell'Expanding Window, per confrontare i risultati in modo coerente. Il 2022 resta una verifica separata, non una guida per il tuning.

In [3]:
# Un caso corrisponde a tutte le 14 × 168 righe long della stessa simulazione.
cases = (
    df[["run_no", "starttime"]]
    .drop_duplicates()
    .sort_values(["starttime", "run_no"])
    .reset_index(drop=True)
)
assert not cases["run_no"].duplicated().any(), "Un run_no non può avere più starttime"

WINDOW_DURATION = pd.Timedelta(hours=168)
TRAIN_WINDOW_YEARS = 3                 # ampiezza della sliding window (anni)
VALIDATION_YEARS = [2019, 2020, 2021]  # 3 fold sliding window
TEST_YEAR = 2022                       # valutazione separata dopo i 3 fold

generator_order = list(pd.unique(df["generator"]))

def get_temporal_split(year, *, limit_validation_to_year=None):
    """Casi nei 3 anni precedenti, senza sovrapposizioni con la validation."""
    if limit_validation_to_year is None:
        limit_validation_to_year = (year != TEST_YEAR)
    val_start = pd.Timestamp(f"{year}-01-01", tz="UTC")
    val_end = pd.Timestamp(f"{year + 1}-01-01", tz="UTC")
    train_start = val_start - pd.DateOffset(years=TRAIN_WINDOW_YEARS)

    # Solo i casi negli ultimi 3 anni la cui finestra termina PRIMA della validation.
    train_cases = cases.loc[
        (cases["starttime"] >= train_start)
        & (cases["starttime"] + WINDOW_DURATION <= val_start)
    ]

    validation_condition = (
        (cases["starttime"] >= val_start)
        & (cases["starttime"] < val_end)
    )
    if limit_validation_to_year:
        # Anche la finestra di validation rimane nell'anno di riferimento.
        validation_condition &= (cases["starttime"] + WINDOW_DURATION <= val_end)

    validation_cases = cases.loc[validation_condition]
    train_ids = set(train_cases["run_no"])
    val_ids = set(validation_cases["run_no"])
    assert train_ids and val_ids, f"Training/validation vuoto per {year}"
    assert train_ids.isdisjoint(val_ids)
    assert train_cases["starttime"].max() + WINDOW_DURATION <= validation_cases["starttime"].min()
    return train_cases, validation_cases, train_ids, val_ids

# Un riepilogo prima di addestrare, per verificare quali anni stiamo usando.
for year in [*VALIDATION_YEARS, TEST_YEAR]:
    train_cases, val_cases, train_ids, val_ids = get_temporal_split(year)
    print(
        f"{year}: train {len(train_ids):,} casi "
        f"({train_cases['starttime'].min().date()} → {train_cases['starttime'].max().date()}) "
        f"| validation {len(val_ids):,} casi "
        f"({val_cases['starttime'].min().date()} → {val_cases['starttime'].max().date()})"
    )

2019: train 1,090 casi (2016-01-01 → 2018-12-25) | validation 359 casi (2019-01-01 → 2019-12-25)
2020: train 1,089 casi (2017-01-01 → 2019-12-25) | validation 360 casi (2020-01-01 → 2020-12-25)
2021: train 1,090 casi (2018-01-01 → 2020-12-25) | validation 359 casi (2021-01-01 → 2021-12-25)
2022: train 1,090 casi (2019-01-01 → 2021-12-25) | validation 365 casi (2022-01-01 → 2022-12-31)


## 4 — Addestramento XGBoost su ciascun fold

**Stesso modello della baseline e del notebook 03:** 200 alberi, profondità 5, learning rate 0.08 e tutte le 27 feature (incluse quelle topologiche). L'unico cambiamento sperimentale è che il training usa una finestra mobile di **3 anni**, invece di tutti gli anni precedenti.

Il ciclo allena **tre modelli indipendenti**; ognuno viene valutato sull'anno successivo e poi liberato dalla memoria. Non scegliamo il migliore dei tre modelli: usiamo le loro metriche per giudicare **la strategia**.

Calcoliamo ROC-AUC micro (su tutte le righe long della validation), accuracy, precision, recall e F1. La media delle ROC-AUC dei fold è un riepilogo, **non** la stessa cosa della ROC-AUC micro sull'unione di tutte le predizioni.

In [4]:
import gc

# Stessi iperparametri usati nella baseline: non stiamo facendo tuning qui.
MODEL_PARAMS = dict(
    objective="binary:logistic",        # ON/OFF
    n_estimators=200,                    # numero di alberi
    max_depth=5,                        # profondità massima
    learning_rate=0.08,                 # contributo di ogni albero
    subsample=0.85,                     # campionamento delle righe
    colsample_bytree=0.85,              # campionamento delle feature
    tree_method="hist",                 # metodo di costruzione degli alberi
    enable_categorical=True,           # generator e plant categoriche
    eval_metric="logloss",
    n_jobs=8,
    random_state=42,
)


def fit_and_evaluate(year, *, keep_predictions=False):
    """Allena XGBoost sugli ultimi 3 anni e valuta sull'anno richiesto."""
    train_cases, val_cases, train_ids, val_ids = get_temporal_split(year)
    train_mask = df["run_no"].isin(train_ids)
    val_mask = df["run_no"].isin(val_ids)

    X_train = df.loc[train_mask, FEATURE_COLUMNS].copy()
    y_train = df.loc[train_mask, "target"].to_numpy(dtype="uint8")
    X_val = df.loc[val_mask, FEATURE_COLUMNS].copy()
    y_val = df.loc[val_mask, "target"].to_numpy(dtype="uint8")
    if len(np.unique(y_train)) < 2 or len(np.unique(y_val)) < 2:
        raise ValueError(f"Training o validation {year} contiene una sola classe")

    print(f"\n=== Validation {year} ===")
    print(f"Casi train: {len(train_ids):,} | righe train: {len(y_train):,}")
    print(f"Casi validation: {len(val_ids):,} | righe validation: {len(y_val):,}")
    print(f"% ON train: {y_train.mean():.2%} | % ON validation: {y_val.mean():.2%}")

    model = XGBClassifier(**MODEL_PARAMS)
    model.set_params(callbacks=[ProgressCallback(model.n_estimators)])
    start = time.perf_counter()
    model.fit(X_train, y_train, verbose=False)
    training_seconds = time.perf_counter() - start

    # Per la ROC-AUC usiamo le probabilità, NON le etichette sogliate.
    probabilities = model.predict_proba(X_val)[:, 1]
    predicted_labels = (probabilities >= 0.5).astype("uint8")
    fold_metrics = {
        "validation_year": int(year),
        "roc_auc_micro": float(roc_auc_score(y_val, probabilities)),
        "accuracy_at_0_5": float(accuracy_score(y_val, predicted_labels)),
        "precision_at_0_5": float(precision_score(y_val, predicted_labels, zero_division=0)),
        "recall_at_0_5": float(recall_score(y_val, predicted_labels, zero_division=0)),
        "f1_at_0_5": float(f1_score(y_val, predicted_labels, zero_division=0)),
        "n_train_cases": int(len(train_ids)),
        "n_validation_cases": int(len(val_ids)),
        "n_train_rows": int(len(y_train)),
        "n_validation_rows": int(len(y_val)),
        "on_train": float(y_train.mean()),
        "on_validation": float(y_val.mean()),
        "train_start": train_cases["starttime"].min().isoformat(),
        "train_end": train_cases["starttime"].max().isoformat(),
        "validation_start": val_cases["starttime"].min().isoformat(),
        "validation_end": val_cases["starttime"].max().isoformat(),
        "training_seconds": round(training_seconds, 2),
    }
    print(f"Training: {training_seconds:.1f} s | ROC-AUC micro: {fold_metrics['roc_auc_micro']:.5f}")
    print("Matrice di confusione [righe = vero, colonne = predetto]:")
    print(confusion_matrix(y_val, predicted_labels, labels=[0, 1]))

    # Conserveremo modello e predizioni soltanto per il test sul 2022.
    val_keys = df.loc[val_mask, ID_COLUMNS].copy().reset_index(drop=True) if keep_predictions else None
    val_probabilities = probabilities.astype("float32") if keep_predictions else None
    del X_train, y_train, X_val, y_val, probabilities, predicted_labels, train_mask, val_mask
    gc.collect()
    if keep_predictions:
        return fold_metrics, model, val_keys, val_probabilities
    del model
    gc.collect()
    return fold_metrics


fold_results = []
for fold_no, validation_year in enumerate(VALIDATION_YEARS, start=1):
    print(f"\n********** FOLD {fold_no}/3 **********")
    fold_results.append(fit_and_evaluate(validation_year))

fold_metrics_df = pd.DataFrame(fold_results)
display(fold_metrics_df[["validation_year", "roc_auc_micro", "n_train_cases", "n_validation_cases", "training_seconds"]])
print(f"ROC-AUC media sui 3 fold: {fold_metrics_df['roc_auc_micro'].mean():.5f}")
print(f"Deviazione standard: {fold_metrics_df['roc_auc_micro'].std(ddof=1):.5f}")


********** FOLD 1/3 **********

=== Validation 2019 ===
Casi train: 1,090 | righe train: 2,563,680
Casi validation: 359 | righe validation: 844,368
% ON train: 56.97% | % ON validation: 51.49%


Training XGBoost:   0%|          | 0/200 [00:00<?, ?albero/s]

Training: 14.3 s | ROC-AUC micro: 0.93699
Matrice di confusione [righe = vero, colonne = predetto]:
[[304475 105110]
 [ 33751 401032]]

********** FOLD 2/3 **********

=== Validation 2020 ===
Casi train: 1,089 | righe train: 2,561,328
Casi validation: 360 | righe validation: 846,720
% ON train: 56.47% | % ON validation: 59.41%


Training XGBoost:   0%|          | 0/200 [00:00<?, ?albero/s]

Training: 14.3 s | ROC-AUC micro: 0.88645
Matrice di confusione [righe = vero, colonne = predetto]:
[[160888 182799]
 [ 25711 477322]]

********** FOLD 3/3 **********

=== Validation 2021 ===
Casi train: 1,090 | righe train: 2,563,680
Casi validation: 359 | righe validation: 844,368
% ON train: 57.88% | % ON validation: 45.39%


Training XGBoost:   0%|          | 0/200 [00:00<?, ?albero/s]

Training: 13.9 s | ROC-AUC micro: 0.90886
Matrice di confusione [righe = vero, colonne = predetto]:
[[363237  97852]
 [ 53092 330187]]


,validation_year,roc_auc_micro,n_train_cases,n_validation_cases,training_seconds
0,2019,0.936993,1090,359,14.27
1,2020,0.886448,1089,360,14.29
2,2021,0.908857,1090,359,13.91


ROC-AUC media sui 3 fold: 0.91077
Deviazione standard: 0.02533


## 5 — Valutazione separata sul 2022 (holdout)

Dopo aver confrontato la sliding window sui fold 2019, 2020 e 2021, facciamo **un quarto addestramento** sui casi del 2019–2021, rispettando lo stesso gap di 168 ore. Valutiamo sul 2022 come controllo più recente.

**Non utilizziamo questo risultato per ritoccare gli hyperparameter**: il 2022 è riservato alla verifica finale. Poiché avevamo già osservato risultati sul 2022 nella baseline e nel notebook expanding, non è un test completamente inesplorato.

Il modello salvato da questo notebook sarà quello addestrato **prima del 2022**, utile a riprodurre la valutazione sull'holdout. In futuro, per prevedere il 2025, alleneremo un **modello definitivo sugli ultimi 3 anni etichettati disponibili** (ad esempio 2020–2022, se l’ultimo anno etichettato è il 2022), oppure su un'altra ampiezza scelta con i fold.

In [5]:
# Quarto modello: allenato sui casi che terminano prima del 2022.
test_metrics, model, test_keys, probabilities = fit_and_evaluate(
    TEST_YEAR, keep_predictions=True
)

print("\nMetriche holdout 2022:")
for name, value in test_metrics.items():
    print(f"{name}: {value}")

print("\nFeature più importanti (importanza XGBoost relativa):")
importance = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)
print(importance.head(15).rename("importance").to_frame())


=== Validation 2022 ===
Casi train: 1,090 | righe train: 2,563,680
Casi validation: 365 | righe validation: 858,480
% ON train: 52.01% | % ON validation: 34.05%


Training XGBoost:   0%|          | 0/200 [00:00<?, ?albero/s]

Training: 13.5 s | ROC-AUC micro: 0.93824
Matrice di confusione [righe = vero, colonne = predetto]:
[[528820  37331]
 [ 60785 231544]]

Metriche holdout 2022:
validation_year: 2022
roc_auc_micro: 0.9382405341418536
accuracy_at_0_5: 0.8857096263162799
precision_at_0_5: 0.861158530915853
recall_at_0_5: 0.7920664730492014
f1_at_0_5: 0.8251687443425207
n_train_cases: 1090
n_validation_cases: 365
n_train_rows: 2563680
n_validation_rows: 858480
on_train: 0.520105863446296
on_validation: 0.34051928990774394
train_start: 2019-01-01T00:00:00+00:00
train_end: 2021-12-25T00:00:00+00:00
validation_start: 2022-01-01T00:00:00+00:00
validation_end: 2022-12-31T00:00:00+00:00
training_seconds: 13.51

Feature più importanti (importanza XGBoost relativa):
                                   importance
price_minus_water_value              0.289626
plant                                0.091298
generator                            0.074175
price_mean_week_minus_water_value    0.052729
min_volume_Staavatn    

## 6 — Riconversione long → wide delle probabilità del test 2022

Riconvertiamo soltanto le **predizioni dell'holdout 2022**, non quelle dei tre fold, in formato wide: una riga per caso e 2.352 colonne `result_committed_<generator>_t<ora>`.

Manteniamo le probabilità ON tra 0 e 1. `total_calculation_time` viene inserito come colonna vuota per compatibilità con il CSV originale; il file serve per ispezionare le predizioni su dati etichettati, **non è una submission per il 2025**.

In [6]:
def predictions_to_wide(
    long_predictions: pd.DataFrame,
    generators: list[str],
    hours: int = 168,
) -> pd.DataFrame:
    """Da (run_no, starttime, generator, hour, prediction) alle colonne wide originali."""
    required = ["run_no", "starttime", "generator", "hour", "prediction"]
    if long_predictions[required].isna().any().any():
        raise ValueError("Chiavi o predizioni mancanti")
    if long_predictions.duplicated(["run_no", "generator", "hour"]).any():
        raise ValueError("Duplicati per (run_no, generator, hour)")
    if not long_predictions["prediction"].between(0, 1).all():
        raise ValueError("Le predizioni devono essere probabilità tra 0 e 1")

    small = long_predictions[required].copy()
    small["target_column"] = (
        "result_committed_" + small["generator"].astype(str)
        + "_t" + small["hour"].astype(str)
    )
    target_columns = [
        f"result_committed_{generator}_t{hour}"
        for generator in generators for hour in range(hours)
    ]

    wide = small.pivot(index=["run_no", "starttime"], columns="target_column", values="prediction")
    missing = set(target_columns) - set(wide.columns)
    if missing:
        raise ValueError(f"Mancano {len(missing)} colonne generatore/ora: {sorted(missing)[:5]}")
    wide = wide.reindex(columns=target_columns).reset_index()
    if wide[target_columns].isna().any().any():
        raise ValueError("Alcuni casi non hanno tutte le 168 ore × tutti i generatori")

    wide = wide.rename(columns={"run_no": "Run No"})
    wide["total_calculation_time"] = np.nan  # Solo placeholder: non disponibile nel long
    return wide[["Run No", "starttime", "total_calculation_time", *target_columns]]


# Associa alle righe del test 2022 le probabilità predette.
long_predictions = test_keys.assign(prediction=probabilities)
wide_predictions = predictions_to_wide(long_predictions, generator_order)
assert len(wide_predictions) == test_metrics["n_validation_cases"]
assert wide_predictions.shape[1] == 2355  # 3 metadata + 14 * 168 target

print(f"Tabella wide: {wide_predictions.shape[0]:,} casi × {wide_predictions.shape[1]:,} colonne")
print("Prime colonne:")
display(wide_predictions.iloc[:3, :9])


Tabella wide: 365 casi × 2,355 colonne
Prime colonne:


target_column,Run No,starttime,total_calculation_time,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2,result_committed_Hogga_G1_t3,result_committed_Hogga_G1_t4,result_committed_Hogga_G1_t5
0,2558,2022-01-01 00:00:00+00:00,NaN,0.121957,0.144359,0.060568,0.062589,0.071492,0.084374
1,2559,2022-01-02 00:00:00+00:00,NaN,0.063053,0.060410,0.059502,0.059502,0.060410,0.063068
2,2560,2022-01-03 00:00:00+00:00,NaN,0.049340,0.048347,0.048347,0.048347,0.049340,0.048347


## 7 — Salvataggio dei risultati (senza sovrascrivere gli altri modelli)

Scriviamo tutto in `Ghigu/data/processed/`, con prefisso **`sliding3y_`**:

- `sliding3y_fold_metrics.csv`: una riga con le metriche di ciascun fold.
- `sliding3y_metrics.json`: riepilogo dei tre fold e metriche holdout 2022.
- `sliding3y_pre2022_xgboost.json`: il modello allenato sui 3 anni precedenti al 2022, **non ancora il modello finale per il 2025**.
- `sliding3y_holdout_2022_predictions_wide.csv`: predizioni sul 2022, nel formato wide.

Il dataset Parquet e tutti i file `baseline_*` e `expanding3_*` esistenti rimangono invariati.

In [7]:
FOLD_METRICS_PATH = PROCESSED_DIR / "sliding3y_fold_metrics.csv"
METRICS_PATH = PROCESSED_DIR / "sliding3y_metrics.json"
MODEL_PATH = PROCESSED_DIR / "sliding3y_pre2022_xgboost.json"
WIDE_PATH = PROCESSED_DIR / "sliding3y_holdout_2022_predictions_wide.csv"

metrics = {
    "cv_strategy": "3-fold sliding window (3y), validation 2019-2021, 168h anti-overlap",
    "train_window_years": TRAIN_WINDOW_YEARS,
    "validation_years": VALIDATION_YEARS,
    "cv_folds": fold_results,
    "cv_mean_roc_auc_micro": float(fold_metrics_df["roc_auc_micro"].mean()),
    "cv_std_roc_auc_micro": float(fold_metrics_df["roc_auc_micro"].std(ddof=1)),
    "holdout_year": TEST_YEAR,
    "holdout_metrics": test_metrics,
    "model_params": {k: v for k, v in MODEL_PARAMS.items()},
}

fold_metrics_df.to_csv(FOLD_METRICS_PATH, index=False)
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
model.save_model(MODEL_PATH)
wide_predictions.to_csv(WIDE_PATH, index=False)

print("Metriche 3 fold:", FOLD_METRICS_PATH)
print("Riepilogo:", METRICS_PATH)
print("Modello pre-2022:", MODEL_PATH)
print("Predizioni wide holdout 2022:", WIDE_PATH)

Metriche 3 fold: /home/ghigu/Scrivania/ml-project-ntnu/Ghigu/data/processed/sliding3y_fold_metrics.csv
Riepilogo: /home/ghigu/Scrivania/ml-project-ntnu/Ghigu/data/processed/sliding3y_metrics.json
Modello pre-2022: /home/ghigu/Scrivania/ml-project-ntnu/Ghigu/data/processed/sliding3y_pre2022_xgboost.json
Predizioni wide holdout 2022: /home/ghigu/Scrivania/ml-project-ntnu/Ghigu/data/processed/sliding3y_holdout_2022_predictions_wide.csv


### Come interpretare i risultati

- Ogni fold ha un **modello addestrato da zero sugli ultimi 3 anni**; la ROC-AUC media permette di confrontare la sliding con la expanding window, usando **gli stessi anni di validation**.
- Il quarto modello è un controllo sul 2022, non un modello definitivo per il 2025. Confrontiamo il risultato con quello del notebook 03, ricordando che il 2022 è già stato osservato.
- La sliding window **non pesa progressivamente i dati per recenza**: esclude semplicemente quelli più vecchi. Gli esempi rimasti hanno pesi uguali.
- Se questa strategia risulterà promettente, potremo provare anche ampiezze differenti (2 o 4 anni) nei fold, e poi allenare un unico modello definitivo usando l'ultima finestra etichettata disponibile.

**Limite ancora da verificare:** oltre a evitare le finestre sovrapposte, dobbiamo essere certi che tutte le feature del long dataset siano effettivamente disponibili al momento della previsione sul 2025, senza utilizzare informazioni future non note.